法布里西奧·卡拉羅 - @fabriciocarraro - https://github.com/fabriciocarraro

##### Copyright 2024 Google LLC。

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# RAG - PDF 在Colab 上使用 Gemma 2 2B 在多個文件中搜尋

本專案示範了使用自然語言處理 (NLP) 技術和 Google 的開源模型 Gemma 2 2B 從Google Colab 上的 PDF 文件中提取、處理和查詢文字資料的管道。系統允許使用者輸入查詢，然後根據 PDF 的內容進行回答。

<table align="left"> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/.archive/Gemma/[Gemma_2]RAG_PDF_Search_in_multiple_documents_on_Colab.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td>
</table>

## 設定
### 選擇 Colab runtime

要完成本教學，您需要擁有 Colab runtime 以及足夠的資源來執行 Gemma 模型。在這種情況下，您可以使用 T4 GPU：
1. 在 Colab 視窗的右上角，選擇 **▾（其他連接選項）**。
2. 選擇**更改 runtime 類型**。
3. 在 **硬體加速器** 下，選擇 **T4 GPU**。

### Gemma Hugging Face 上的 2 個設置

本食譜使用Gemma 2B 指令透過Hugging Face調整模型。所以你需要：
* 接受特定型號的Hugging Face頁面上的Gemma 2許可證，即可存取[huggingface.co](huggingface.co)上的Gemma 2，即[Gemma 2B IT](https://huggingface.co/google/gemma-2-2b-it@P0011)。
* 產生 [Hugging Face access token](https://huggingface.co/docs/hub/en/security-tokens) 並設定為 Colab secret 'HF_TOKEN'。

## 檢索增強生成 (RAG)

大型语言模型 (LLM) 无需直接接受训练即可学习新能力。然而，众所周知，法学硕士在回答他们未经培训的问题时会产生“幻觉”。部分原因是法學碩士不知道訓練後發生的事件。追蹤法學碩士的回覆來源也非常困難。对于可靠、可扩展的应用程序，法学硕士提供基于事实的答复并能够引用其信息来源非常重要。
用於克服這些限制的常用方法稱為檢索增強生成（RAG），它使用通過資訊檢索（IR）機制從外部知識庫檢索的相關數據來增強發送到法學碩士的prompt。知識庫可以是您自己的文件、資料庫或APIs 語料庫。

## 安裝並導入相依性

In [ ]:
!pip install -q transformers sentence_transformers faiss-cpu torch PyPDF2 nltk

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd
import PyPDF2
import os
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize
from google.colab import userdata

## 設定模型和tokenizer

In [ ]:
HUGGING_FACE_ACCESS_TOKEN = userdata.get('HF_TOKEN')

model_name = 'google/gemma-2-2b-it'

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    token=HUGGING_FACE_ACCESS_TOKEN
    ).to('cuda')

tokenizer = AutoTokenizer.from_pretrained(model_name, token=HUGGING_FACE_ACCESS_TOKEN)

## 從 PDF 文件中提取和token化訊息

`extract_text_from_pdf()` 函數將尋找 `pdf_path` 資料夾中的所有 PDF 檔案。
`split_text_into_chunks()` 函數取得文字並將其分解為更小的區塊。

In [ ]:
def extract_text_from_pdf(pdf_path):
    try:
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            text = "".join([page.extract_text() for page in reader.pages])
        return text
    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")
        return ""

def split_text_into_chunks(text, max_chunk_size=1000):
    sentences = sent_tokenize(text)
    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= max_chunk_size:
            current_chunk += sentence + " "
        else:
            chunks.append(current_chunk.strip())
            current_chunk = sentence + " "

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

## 從 PDF 中提取信息

將變數 `pdf_directory` 設定為 PDF 檔案所在的路徑。在這種情況下，在 Google Colab 上執行，它們將位於名為 `PDFs` 的資料夾中，該資料夾可以在左側的內容區域中建立。
建立一個 Pandas DataFrame，其中包含對應 PDF 的路徑、其區塊及其區塊的嵌入向量。

In [ ]:
encoder = SentenceTransformer('all-MiniLM-L6-v2')

# Process PDF files
pdf_directory = "/content/PDFs/"
df_documents = pd.DataFrame(columns=['path', 'text_chunks', 'embeddings'])

for filename in os.listdir(pdf_directory):
    if filename.endswith(".pdf"):
        print(filename)
        pdf_path = os.path.join(pdf_directory, filename)
        text = extract_text_from_pdf(pdf_path)
        chunks = split_text_into_chunks(text)
        document_embeddings = encoder.encode(chunks)
        new_row = pd.DataFrame({'path': [pdf_path], 'text_chunks': [chunks], 'embeddings': [document_embeddings]})
        df_documents = pd.concat([df_documents, new_row], ignore_index=True)

df_documents

## 從所有文件嵌入建立 FAISS 索引

Faiss 是一個用於高效相似性搜尋和向量聚類的library。 IndexFlatL2 演算法將應用於所有區塊嵌入向量。

In [ ]:
all_embeddings = np.vstack(df_documents['embeddings'].tolist())
dimension = all_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(all_embeddings)

## 計算嵌入距離並產生答案

`find_most_similar_chunks()` 函數將為您的查詢建立嵌入向量，並將其與從 PDF 文件檢索到的所有區塊的相似性進行比較，傳回最相似的一個，該向量將用作下一個函數的上下文。
`generate_response()` 函數將根據從最相似的資訊區塊檢索的上下文，使用我們選擇的模型 (Gemma 2 2B) 產生答案。

In [ ]:
def find_most_similar_chunks(query, top_k=3):
    query_embedding = encoder.encode([query])
    distances, indices = index.search(query_embedding, top_k)
    results = []
    total_chunks = sum(len(chunks) for chunks in df_documents['text_chunks'])
    for i, idx in enumerate(indices[0]):
        if idx < total_chunks:
            doc_idx = 0
            chunk_idx = idx
            while chunk_idx >= len(df_documents['text_chunks'].iloc[doc_idx]):
                chunk_idx -= len(df_documents['text_chunks'].iloc[doc_idx])
                doc_idx += 1
            results.append({
                'document': df_documents['path'].iloc[doc_idx],
                'chunk': df_documents['text_chunks'].iloc[doc_idx][chunk_idx],
                'distance': distances[0][i]
            })
    return results

def generate_response(query, context, max_length=1000):
    prompt = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to('cuda')

    with torch.no_grad():
        output = model.generate(input_ids, max_new_tokens=max_length, num_return_sequences=1)

    decoded_output = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extracting the answer part by removing the prompt portion
    answer_start = decoded_output.find("Answer:") + len("Answer:")
    answer = decoded_output[answer_start:].strip()

    return answer

def query_documents(query):
    similar_chunks = find_most_similar_chunks(query)
    context = " ".join([result['chunk'].replace("\n", "") for result in similar_chunks])
    response = generate_response(query, context)
    return response, similar_chunks

## 在 PDF 中尋找信息

The variable `query` contains the information you want to retrieve from the PDF files.

In [ ]:
query = "How many types of regular Train Car cards are there?"
answer, relevant_chunks = query_documents(query)

print(f"Query: {query}\n\n-----\n")
print(f"Generated answer: {answer}\n\n-----\n")
print("Relevant chunks:")
for chunk in relevant_chunks:
    print(f"Document: {chunk['document']}")
    print(f"Chunk: {chunk['chunk']}".replace("\n", ""))
    print(f"Distance: {chunk['distance']}")
    print()